# NDgpu — time-dependent S_N transport on GPU (Colab)

Phase-2 GPU check for `TransientSNSolver` (2D Cartesian). Backward Euler puts
`theta = 1/(v dt)` on the collision diagonal and a **per-ordinate** time source
`theta*psi_old` on the right-hand side, so the transient carries the full angular
flux between steps. Both now ride the **vectorized wavefront sweep**, which means
the whole per-step transport solve is a fixed sequence of batched numpy/cupy
kernels — the same structure that made the steady solver a GPU win.

Three things this notebook measures:

1. **CPU vs GPU** across mesh refinement — where the transient crossover sits.
2. **`step_acceleration`** (`"cmfd"` vs `"none"`): the drift-corrected coarse
   solve cuts fixed-point iterations per step. It is a *host* sparse-LU solve, so
   on GPU it is the Amdahl term — the interesting question is whether the CMFD win
   (fewer sweeps) still beats its host cost on device.
3. **dt sensitivity** — the theta shift damps the within-step fixed point, so
   small steps converge in fewer iterations and CMFD matters less.

CPU and GPU run the identical iteration sequence, so the reported speed-ups are
tolerance-independent; `dpcm` and `dpow` confirm the answers match.

Upload `dist/ndgpu-src.zip` when prompted. Set `NDGPU_QUICK=1` for a smoke run.

In [ ]:
import os
try:                                        # Colab: upload dist/ndgpu-src.zip
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    get_ipython().run_line_magic("pip", f"install -q {zip_name}")
    try:
        import cupy
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q cupy-cuda12x")
    get_ipython().system("nvidia-smi -L")
except ImportError:                         # local run: ndgpu already importable
    pass

In [ ]:
import time
import numpy as np
from ndgpu import Grid, Kinetics, Material, TransientSNSolver

try:
    import cupy
    HAVE_GPU = cupy.cuda.runtime.getDeviceCount() > 0
except Exception:
    HAVE_GPU = False
print("GPU available:", HAVE_GPU)

QUICK = bool(os.environ.get("NDGPU_QUICK"))

D, SIGMA_A, NU_SIGMA_F = 1.3, 0.030, 0.035
V, BETA, LAM = 2.2e5, 0.0065, 0.08
BASE = Material(name="1g", diffusion=[D], sigma_a=[SIGMA_A],
                nu_sigma_f=[NU_SIGMA_F])
KIN = Kinetics(velocities=[V], beta=[BETA], decay=[LAM])
RHO_DOLLARS = 0.5
EPS = RHO_DOLLARS * BETA / (1.0 - RHO_DOLLARS * BETA)

# Angular refinement drives the transient's memory: psi is (M, nx, ny) per group
# and is retained between steps. S16 (M = 2*16) on 96^2 is ~2.4 MB/group.
QUAD = dict(n_polar=2, n_azi=16, acceleration="dsa")
NCELLS = [24, 40] if QUICK else [24, 40, 64, 96]
N_STEPS = 6 if QUICK else 20


def problem(eps):
    pert = [Material(name="pert", diffusion=[D], sigma_a=[SIGMA_A],
                     nu_sigma_f=[NU_SIGMA_F * (1.0 + eps)])]
    return lambda t: (([BASE] if t <= 0 else pert), None)


def run(n, device, accel, dt):
    grid = Grid(shape=(n, n, 1), size=(120.0, 120.0, 1.0))
    solver = TransientSNSolver(grid, problem(EPS), KIN, bc="vacuum",
                               device=device, step_acceleration=accel, **QUAD)
    t0 = time.perf_counter()
    res = solver.solve(t_end=N_STEPS * dt, dt=dt, tol_step=1e-8)
    return dict(k=res.k0, power=res.power[-1], sweeps=res.total_inner_iterations,
                its=float(np.mean(res.step_iterations)),
                solve=time.perf_counter() - t0, M=solver._engine(
                    *solver.problem_at(0.0)).M)

## 1. CPU vs GPU across mesh refinement (CMFD on, dt = 1e-3 s)

In [ ]:
DT = 1e-3
rows = []
for n in NCELLS:
    cpu = run(n, "cpu", "cmfd", DT)
    row = dict(n=n, **cpu)
    if HAVE_GPU:
        gpu = run(n, "gpu", "cmfd", DT)
        row.update(t_gpu=gpu["solve"], speedup=cpu["solve"] / gpu["solve"],
                   dpcm=(gpu["k"] - cpu["k"]) * 1e5,
                   dpow=abs(gpu["power"] - cpu["power"]))
    rows.append(row)

hdr = f"{'ncell':>6} {'cells':>7} {'M':>4} {'its/step':>9} {'sweeps':>7} {'k0':>9} {'t_cpu[s]':>9}"
if HAVE_GPU:
    hdr += f" {'t_gpu[s]':>9} {'speedup':>8} {'dpcm':>7} {'dpow':>9}"
print(hdr)
for r in rows:
    line = (f"{r['n']:>6d} {r['n']**2:>7d} {r['M']:>4d} {r['its']:>9.2f} "
            f"{r['sweeps']:>7d} {r['k']:>9.6f} {r['solve']:>9.2f}")
    if HAVE_GPU:
        line += (f" {r['t_gpu']:>9.2f} {r['speedup']:>7.2f}x {r['dpcm']:>7.2f}"
                 f" {r['dpow']:>9.1e}")
    print(line)

## 2. CMFD step acceleration — does the host coarse solve still pay on GPU?

`step_acceleration="cmfd"` builds a drift-corrected diffusion operator (one
current-accumulating sweep per group) and solves the coarse fixed point with
**host** sparse LU. It cuts fixed-point iterations per step several-fold on CPU.
On GPU the sweeps get cheaper while the coarse solve does not, so this is the
Amdahl term — the same host-bound CMFD ceiling documented for the steady solver.

In [ ]:
devs = ["cpu"] + (["gpu"] if HAVE_GPU else [])
print(f"{'ncell':>6} {'device':>7} {'accel':>6} {'its/step':>9} {'sweeps':>7} "
      f"{'t[s]':>8} {'vs none':>8}")
for n in NCELLS:
    for dev in devs:
        base = None
        for accel in ("none", "cmfd"):
            r = run(n, dev, accel, DT)
            if base is None:
                base = r["solve"]
            print(f"{n:>6d} {dev:>7} {accel:>6} {r['its']:>9.2f} "
                  f"{r['sweeps']:>7d} {r['solve']:>8.2f} {base / r['solve']:>7.2f}x")

## 3. dt sensitivity — the theta shift is free preconditioning

`theta = 1/(v dt)` is added to the total cross section, so a *small* time step
makes the within-group problem strongly diagonally dominant and the within-step
fission fixed point contract fast. As dt grows the step approaches the steady
eigenproblem and the iteration count climbs — which is exactly where CMFD earns
its keep. (CPU reference measurement, 40x40: 6.0 its/step at dt = 1e-4 s rising
to 12.3 at dt = 5e-2 s without CMFD; ~3-4 with it at every dt.)

In [ ]:
DTS = [1e-4, 1e-3] if QUICK else [1e-4, 1e-3, 1e-2, 5e-2]
n = NCELLS[1]
print(f"{'dt[s]':>8} {'theta':>9} {'accel':>6} {'its/step':>9} {'sweeps':>7} {'t[s]':>8}")
for dt in DTS:
    for accel in ("none", "cmfd"):
        r = run(n, devs[-1], accel, dt)
        print(f"{dt:>8.0e} {1.0 / (V * dt):>9.5f} {accel:>6} {r['its']:>9.2f} "
              f"{r['sweeps']:>7d} {r['solve']:>8.2f}")

### Reading the tables

* **Table 1** — the transient crossover. Each time step is several within-group
  transport solves, so per-solve GPU efficiency matters more than in the steady
  case (which amortises setup over few outers). `dpcm`/`dpow` must be ~0.
* **Table 2** — if `cmfd` still wins on GPU, the host coarse solve is affordable;
  if the GPU `cmfd`/`none` ratio is much worse than the CPU one, the coarse solve
  has become the bottleneck and wants the device-multigrid treatment already
  built for the 3D tri solver.
* **Table 3** — picks the step size at which acceleration is worth enabling.
  Prompt transients (small dt) barely need it; delayed-phase marches do.